# BayesNAM

## imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
import math

## BayesLinear + BayesMLP

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.nn import Parameter

class BayesLinear(nn.Module):
    def __init__(self, in_features, out_features,
                 bias=True,
                 weight_prior_mu=0.0, weight_prior_sigma=0.1,
                 bias_prior_mu=0.0, bias_prior_sigma=0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.use_bias = bias

        self.weight_mu = Parameter(torch.Tensor(out_features, in_features).uniform_(-0.5, 0.5))
        self.weight_logsigma = Parameter(torch.full((out_features, in_features), math.log(0.1)))

        if self.use_bias:
            self.bias_mu = Parameter(torch.Tensor(out_features).uniform_(-0.1, 0.1))
            self.bias_logsigma = Parameter(torch.full((out_features,), math.log(0.1)))
        else:
            self.register_parameter('bias_mu', None)
            self.register_parameter('bias_logsigma', None)

        self.register_buffer('w_prior_mu', torch.tensor(weight_prior_mu))
        self.register_buffer('w_prior_logsigma', torch.tensor(math.log(weight_prior_sigma)))
        if self.use_bias:
            self.register_buffer('b_prior_mu', torch.tensor(bias_prior_mu))
            self.register_buffer('b_prior_logsigma', torch.tensor(math.log(bias_prior_sigma)))

    @staticmethod
    def kl_divergence(mu_q, log_sigma_q, mu_p, log_sigma_p):
        sigma_q = torch.exp(log_sigma_q)
        sigma_p = torch.exp(log_sigma_p)
        term1 = log_sigma_p - log_sigma_q
        term2 = (sigma_q.pow(2) + (mu_q - mu_p).pow(2)) / (2.0 * sigma_p.pow(2))
        kl = term1 + term2 - 0.5
        return kl.sum()

    def forward(self, x):
        eps_w = torch.randn_like(self.weight_mu)
        weight_sigma = torch.exp(self.weight_logsigma)
        W = self.weight_mu + weight_sigma * eps_w

        if self.use_bias:
            eps_b = torch.randn_like(self.bias_mu)
            bias_sigma = torch.exp(self.bias_logsigma)
            b = self.bias_mu + bias_sigma * eps_b
        else:
            b = None

        out = F.linear(x, W, b)

        kl_w = self.kl_divergence(self.weight_mu, self.weight_logsigma,
                                  self.w_prior_mu, self.w_prior_logsigma)
        if self.use_bias:
            kl_b = self.kl_divergence(self.bias_mu, self.bias_logsigma,
                                      self.b_prior_mu, self.b_prior_logsigma)
            kl = kl_w + kl_b
        else:
            kl = kl_w

        return out, kl

rng = np.random.default_rng(42)
n_train = 300
X_train = rng.uniform(-3.5, 3.5, size=(n_train, 1)).astype(np.float32)
y_train = np.sin(X_train) + 0.2 * rng.normal(size=(n_train, 1)).astype(np.float32)
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)

class BayesMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = BayesLinear(1, 50)
        self.b2 = BayesLinear(50, 50)
        self.b3 = BayesLinear(50, 1)

    def forward(self, x):
        out1, kl1 = self.b1(x)
        act1 = torch.relu(out1)
        out2, kl2 = self.b2(act1)
        act2 = torch.relu(out2)
        out3, kl3 = self.b3(act2)
        return out3, (kl1 + kl2 + kl3)
    
    def elbo_loss(self, x, y, noise_sigma=1.0, n_samples=1):
        if y.dim() == 1:
            y = y.unsqueeze(-1)
        
        log_likelihoods = []
        kl_divs = []
        
        for _ in range(n_samples):
            pred, kl = self.forward(x)
            
            mse = F.mse_loss(pred, y, reduction='sum')
            n_data = y.size(0)
            log_lik = -0.5 * mse / (noise_sigma ** 2) \
                      - 0.5 * n_data * math.log(2 * math.pi * noise_sigma ** 2)
            
            log_likelihoods.append(log_lik)
            kl_divs.append(kl)
        
        mean_log_lik = torch.mean(torch.stack(log_likelihoods))
        mean_kl = torch.mean(torch.stack(kl_divs))
        
        elbo = mean_log_lik - mean_kl
        loss = -elbo
        
        return loss, mean_log_lik, mean_kl, elbo

model = BayesMLP()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)

def train_epoch(model, x, y, batch_size=32, kl_weight=0.01):
    model.train()
    n = x.size(0)
    perm = torch.randperm(n)
    total_loss = 0.0
    total_data = 0.0
    total_kl = 0.0
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = x[idx], y[idx]
        pred, kl = model(xb)
        data_loss = F.mse_loss(pred, yb)
        loss = data_loss + kl_weight * kl / n

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item() * xb.size(0)
        total_data += data_loss.item() * xb.size(0)
        total_kl += kl.item()
    return total_loss / n, total_data / n, total_kl / n

def train_epoch_elbo(model, optimizer, x, y, noise_sigma=1.0, n_samples=1):
    model.train()
    optimizer.zero_grad()
    
    loss, log_lik, kl, elbo = model.elbo_loss(x, y, noise_sigma=noise_sigma, n_samples=n_samples)
    
    loss.backward()
    optimizer.step()
    
    return loss.item(), log_lik.item(), kl.item(), elbo.item()

def train_with_elbo(model, x_train, y_train, x_test, y_test, 
                    epochs=1000, lr=1e-3, noise_sigma=1.0, 
                    n_samples=1, print_every=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'loss': [], 'log_lik': [], 'kl': [], 'elbo': [],
        'test_mse': []
    }
    
    for epoch in range(epochs):
        loss, log_lik, kl, elbo = train_epoch_elbo(
            model, optimizer, x_train, y_train, 
            noise_sigma=noise_sigma, n_samples=n_samples
        )
        
        model.eval()
        with torch.no_grad():
            y_pred, _ = model(x_test)
            test_mse = F.mse_loss(y_pred, y_test).item()
        
        history['loss'].append(loss)
        history['log_lik'].append(log_lik)
        history['kl'].append(kl)
        history['elbo'].append(elbo)
        history['test_mse'].append(test_mse)
        
        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"Loss: {loss:.4f} | "
                  f"Log-Lik: {log_lik:.4f} | "
                  f"KL: {kl:.4f} | "
                  f"ELBO: {elbo:.4f} | "
                  f"Test MSE: {test_mse:.4f}")
    
    return history

history = []
print("Training started...")
for epoch in range(500):
    loss, data_l, kl_l = train_epoch(model, X_train_t, y_train_t, batch_size=32, kl_weight=0.01)
    history.append((loss, data_l, kl_l))
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Loss={loss:.4f}, Data Loss={data_l:.4f}, KL={kl_l:.4f}")

print("Training completed!")

fig1 = plt.figure(figsize=(10, 4))
loss_arr = np.array(history)
plt.plot(loss_arr[:,0], label="total loss")
plt.plot(loss_arr[:,1], label="data loss (MSE)")
plt.plot(loss_arr[:,2] * 0.01, label="KL*weight (per sample)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Training: ELBO components")
plt.grid(alpha=0.3)

model.eval()
xs = np.linspace(-4.5, 4.5, 300, dtype=np.float32).reshape(-1,1)
xs_t = torch.from_numpy(xs)

with torch.no_grad():
    S = 100
    preds_accum = []
    for _ in range(S):
        preds, _ = model(xs_t)
        preds_accum.append(preds.numpy())
    preds_accum = np.stack(preds_accum, axis=0).squeeze(-1)
    mean_pred = preds_accum.mean(axis=0)
    std_pred = preds_accum.std(axis=0)

fig2 = plt.figure(figsize=(10, 6))
xs_true = xs.squeeze()
y_true = np.sin(xs_true)
plt.plot(xs_true, y_true, 'g--', linewidth=2, label="true function: sin(x)", alpha=0.7)

plt.scatter(X_train.squeeze(), y_train.squeeze(), s=20, alpha=0.8, color='blue', label="train data", zorder=3)

plt.plot(xs.squeeze(), mean_pred, 'r-', linewidth=2.5, label="predictive mean", zorder=2)

upper = mean_pred + 2*std_pred
lower = mean_pred - 2*std_pred
plt.fill_between(xs.squeeze(), lower, upper, alpha=0.3, color='orange', label="±2 std (95% CI)", zorder=1)

plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.title("Bayesian NN: Predictive Mean and Uncertainty", fontsize=14)
plt.legend(loc="upper right", fontsize=10)
plt.grid(alpha=0.3)
plt.ylim(-2.5, 2.5)

fig3 = plt.figure(figsize=(10, 4))
with torch.no_grad():
    w_mu = model.b1.weight_mu.view(-1)
    w_logsig = model.b1.weight_logsigma.view(-1)
    for idx in [0, 1, 2]:
        eps = torch.randn(2000)
        sigma = torch.exp(w_logsig[idx])
        samples_w = (w_mu[idx] + sigma * eps).numpy()
        plt.hist(samples_w, bins=40, alpha=0.5, label=f"W[{idx}] (μ={w_mu[idx]:.3f}, σ={sigma:.3f})")
plt.xlabel("Weight value", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("Posterior samples of selected weights (Layer 1)", fontsize=14)
plt.legend(loc="best", fontsize=10)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal MSE on training data: {loss_arr[-1, 1]:.4f}")
print("\nELBO functions added to BayesMLP class")

In [ ]:
class BayesMLP_Flexible(nn.Module):
    def __init__(self, in_features=1, out_features=1, hidden_units=[50, 50],
                 weight_prior_mu=0.0, weight_prior_sigma=0.1,
                 bias_prior_mu=0.0, bias_prior_sigma=0.1,
                 activation=nn.ReLU()):
        super().__init__()
        self.activation = activation
        
        layer_sizes = [in_features] + list(hidden_units) + [out_features]
        self.layers = nn.ModuleList()
        
        for i in range(len(layer_sizes) - 1):
            layer = BayesLinear(
                in_features=layer_sizes[i], 
                out_features=layer_sizes[i + 1],
                bias=True,
                weight_prior_mu=weight_prior_mu,
                weight_prior_sigma=weight_prior_sigma,
                bias_prior_mu=bias_prior_mu,
                bias_prior_sigma=bias_prior_sigma
            )
            self.layers.append(layer)
    
    def forward(self, x):
        kl_total = 0.0
        
        for layer in self.layers[:-1]:
            x, kl = layer(x)
            kl_total += kl
            x = self.activation(x)
        
        x, kl = self.layers[-1](x)
        kl_total += kl
        
        return x, kl_total
    
    def elbo_loss(self, x, y, noise_sigma=1.0, n_samples=1):
        if y.dim() == 1:
            y = y.unsqueeze(-1)
        
        log_likelihoods = []
        kl_divs = []
        
        for _ in range(n_samples):
            pred, kl = self.forward(x)
            
            mse = F.mse_loss(pred, y, reduction='sum')
            n_data = y.size(0)
            log_lik = -0.5 * mse / (noise_sigma ** 2) \
                      - 0.5 * n_data * math.log(2 * math.pi * noise_sigma ** 2)
            
            log_likelihoods.append(log_lik)
            kl_divs.append(kl)
        
        mean_log_lik = torch.mean(torch.stack(log_likelihoods))
        mean_kl = torch.mean(torch.stack(kl_divs))
        
        elbo = mean_log_lik - mean_kl
        loss = -elbo
        
        return loss, mean_log_lik, mean_kl, elbo

print("BayesMLP_Flexible class with ELBO method defined")

In [ ]:
class BayesNAM(nn.Module):
    def __init__(self, n_features, hidden_units=[50, 50],
                 weight_prior_mu=0.0, weight_prior_sigma=0.1,
                 bias_prior_mu=0.0, bias_prior_sigma=0.1):
        super().__init__()
        self.n_features = n_features
        self.sub_nets = nn.ModuleList()
        
        for _ in range(n_features):
            mlp = BayesMLP_Flexible(
                in_features=1,
                out_features=1,
                hidden_units=hidden_units,
                weight_prior_mu=weight_prior_mu,
                weight_prior_sigma=weight_prior_sigma,
                bias_prior_mu=bias_prior_mu,
                bias_prior_sigma=bias_prior_sigma
            )
            self.sub_nets.append(mlp)
        
        self.bias_mu = Parameter(torch.zeros(1))
        self.bias_logsigma = Parameter(torch.full((1,), math.log(0.1)))
        self.register_buffer('bias_prior_mu', torch.tensor(bias_prior_mu))
        self.register_buffer('bias_prior_logsigma', torch.tensor(math.log(bias_prior_sigma)))

    def forward(self, x):
        if isinstance(x, tuple) or isinstance(x, list):
            assert len(x) == self.n_features, "Input feature count mismatch."
            feature_tensors = x
        else:
            assert x.size(1) == self.n_features, "Input feature count mismatch."
            feature_tensors = [x[:, i:i+1] for i in range(self.n_features)]

        outputs = []
        total_kl = 0.0
        
        for i in range(self.n_features):
            out, kl = self.sub_nets[i](feature_tensors[i])
            outputs.append(out)
            total_kl += kl
        
        eps_b = torch.randn_like(self.bias_mu)
        bias_sigma = torch.exp(self.bias_logsigma)
        bias = self.bias_mu + bias_sigma * eps_b
        
        bias_kl = BayesLinear.kl_divergence(
            self.bias_mu, self.bias_logsigma,
            self.bias_prior_mu, self.bias_prior_logsigma
        )
        total_kl += bias_kl
        
        final_output = torch.sum(torch.stack(outputs, dim=0), dim=0) + bias
        
        return final_output, total_kl
    
    def get_feature_contributions(self, x, n_samples=100):
        self.eval()
        if isinstance(x, tuple) or isinstance(x, list):
            feature_tensors = x
        else:
            feature_tensors = [x[:, i:i+1] for i in range(self.n_features)]
        
        all_contributions = []
        
        with torch.no_grad():
            for _ in range(n_samples):
                feature_outputs = []
                for i in range(self.n_features):
                    out, _ = self.sub_nets[i](feature_tensors[i])
                    feature_outputs.append(out)
                all_contributions.append(torch.cat(feature_outputs, dim=1))
            
            mean_contributions = torch.stack(all_contributions).mean(dim=0)
        
        return mean_contributions
    
    def elbo_loss(self, x, y, noise_sigma=1.0, n_samples=1):
        if y.dim() == 1:
            y = y.unsqueeze(-1)
        
        log_likelihoods = []
        kl_divs = []
        
        for _ in range(n_samples):
            pred, kl = self.forward(x)
            
            mse = F.mse_loss(pred, y, reduction='sum')
            n_data = y.size(0)
            log_lik = -0.5 * mse / (noise_sigma ** 2) \
                      - 0.5 * n_data * math.log(2 * math.pi * noise_sigma ** 2)
            
            log_likelihoods.append(log_lik)
            kl_divs.append(kl)
        
        mean_log_lik = torch.mean(torch.stack(log_likelihoods))
        mean_kl = torch.mean(torch.stack(kl_divs))
        
        elbo = mean_log_lik - mean_kl
        loss = -elbo
        
        return loss, mean_log_lik, mean_kl, elbo

print("BayesNAM class with ELBO method defined")

## Test BayesNAM with ELBO Training (2 Features)

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

n_train = 100
n_test = 100
n_features = 2

X_train = torch.randn(n_train, n_features)
y_train = (X_train[:, 0]**2 + torch.sin(2 * X_train[:, 1]) + 0.1 * torch.randn(n_train)).unsqueeze(-1)

X_test = torch.randn(n_test, n_features)
y_test = (X_test[:, 0]**2 + torch.sin(2 * X_test[:, 1]) + 0.1 * torch.randn(n_test)).unsqueeze(-1)

print(f"Training data: X shape {X_train.shape}, y shape {y_train.shape}")
print(f"Test data: X shape {X_test.shape}, y shape {y_test.shape}")
print(f"\nTrue function: y = x1^2 + sin(2*x2) + noise")

model_nam = BayesNAM(n_features=2, hidden_units=[30, 30])
print(f"\nBayesNAM model initialized with {n_features} features")

In [ ]:
print("Training BayesNAM with ELBO loss...\n")

optimizer = torch.optim.Adam(model_nam.parameters(), lr=5e-3)

epochs = 1000
print_every = 50
history = {'loss': [], 'log_lik': [], 'kl': [], 'elbo': [], 'test_mse': []}

for epoch in range(epochs):
    model_nam.train()
    optimizer.zero_grad()
    
    loss, log_lik, kl, elbo = model_nam.elbo_loss(
        X_train, y_train, 
        noise_sigma=0.1,
        n_samples=1
    )
    
    loss.backward()
    optimizer.step()
    
    model_nam.eval()
    with torch.no_grad():
        y_pred, _ = model_nam(X_test)
        test_mse = F.mse_loss(y_pred, y_test).item()
    
    history['loss'].append(loss.item())
    history['log_lik'].append(log_lik.item())
    history['kl'].append(kl.item())
    history['elbo'].append(elbo.item())
    history['test_mse'].append(test_mse)
    
    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Loss: {loss.item():.4f} | "
              f"Log-Lik: {log_lik.item():.4f} | "
              f"KL: {kl.item():.4f} | "
              f"ELBO: {elbo.item():.4f} | "
              f"Test MSE: {test_mse:.4f}")

print("\nTraining completed!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(history['loss'], color='red', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (-ELBO)')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['elbo'], color='green', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('ELBO')
axes[0, 1].set_title('Evidence Lower Bound')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['log_lik'], label='Log-Likelihood', color='blue', linewidth=2)
axes[1, 0].plot(history['kl'], label='KL Divergence', color='orange', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Value')
axes[1, 0].set_title('ELBO Components')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['test_mse'], color='purple', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('MSE')
axes[1, 1].set_title('Test Performance')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"   ELBO: {history['elbo'][-1]:.4f}")
print(f"   Log-Likelihood: {history['log_lik'][-1]:.4f}")
print(f"   KL Divergence: {history['kl'][-1]:.4f}")
print(f"   Test MSE: {history['test_mse'][-1]:.4f}")

In [ ]:
model_nam.eval()

n_mc_samples = 100
all_contributions = []

with torch.no_grad():
    for _ in range(n_mc_samples):
        feature_outputs = []
        feature_tensors = [X_test[:, i:i+1] for i in range(2)]
        for i in range(2):
            out, _ = model_nam.sub_nets[i](feature_tensors[i])
            feature_outputs.append(out)
        all_contributions.append(torch.cat(feature_outputs, dim=1))

all_contributions = torch.stack(all_contributions)
mean_contributions = all_contributions.mean(dim=0)
std_contributions = all_contributions.std(dim=0)

n_grid = 50
x1_grid = torch.linspace(X_test[:, 0].min(), X_test[:, 0].max(), n_grid).unsqueeze(-1)
x2_grid = torch.linspace(X_test[:, 1].min(), X_test[:, 1].max(), n_grid).unsqueeze(-1)

with torch.no_grad():
    grid_contrib_1_all = []
    grid_contrib_2_all = []
    for _ in range(n_mc_samples):
        out1, _ = model_nam.sub_nets[0](x1_grid)
        out2, _ = model_nam.sub_nets[1](x2_grid)
        grid_contrib_1_all.append(out1)
        grid_contrib_2_all.append(out2)
    
    grid_contrib_1_mean = torch.stack(grid_contrib_1_all).mean(dim=0).squeeze()
    grid_contrib_1_std = torch.stack(grid_contrib_1_all).std(dim=0).squeeze()
    grid_contrib_2_mean = torch.stack(grid_contrib_2_all).mean(dim=0).squeeze()
    grid_contrib_2_std = torch.stack(grid_contrib_2_all).std(dim=0).squeeze()

torch.manual_seed(43)
x1_data = torch.linspace(X_test[:, 0].min(), X_test[:, 0].max(), 40)
x2_data = torch.linspace(X_test[:, 1].min(), X_test[:, 1].max(), 40)
y1_true = x1_data ** 2
y2_true = torch.sin(2 * x2_data)
y1_noisy = y1_true + 0.1 * torch.randn(40)
y2_noisy = y2_true + 0.1 * torch.randn(40)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x1_data.numpy(), y1_noisy.numpy(),
                alpha=0.4, s=20, color='darkblue', label='Data Points (with noise)', zorder=2)
axes[0].plot(x1_grid.squeeze().numpy(), grid_contrib_1_mean.numpy(), 
             'b-', linewidth=2.5, label='Learned Mean', zorder=3)
axes[0].fill_between(x1_grid.squeeze().numpy(),
                     (grid_contrib_1_mean - 2*grid_contrib_1_std).numpy(),
                     (grid_contrib_1_mean + 2*grid_contrib_1_std).numpy(),
                     alpha=0.3, color='blue', label='95% CI (Epistemic)', zorder=1)
axes[0].plot(x1_grid.squeeze().numpy(), x1_grid.squeeze().numpy()**2, 
             'r--', linewidth=2, label='True: x1²', zorder=4)
axes[0].set_xlabel('Feature x1', fontsize=12)
axes[0].set_ylabel('Contribution to y', fontsize=12)
axes[0].set_title('Feature 1 Contribution with 95% CI', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(x2_data.numpy(), y2_noisy.numpy(),
                alpha=0.4, s=20, color='darkgreen', label='Data Points (with noise)', zorder=2)
axes[1].plot(x2_grid.squeeze().numpy(), grid_contrib_2_mean.numpy(), 
             'g-', linewidth=2.5, label='Learned Mean', zorder=3)
axes[1].fill_between(x2_grid.squeeze().numpy(),
                     (grid_contrib_2_mean - 2*grid_contrib_2_std).numpy(),
                     (grid_contrib_2_mean + 2*grid_contrib_2_std).numpy(),
                     alpha=0.3, color='green', label='95% CI (Epistemic)', zorder=1)
axes[1].plot(x2_grid.squeeze().numpy(), torch.sin(2 * x2_grid.squeeze()).numpy(), 
             'r--', linewidth=2, label='True: sin(2x2)', zorder=4)
axes[1].set_xlabel('Feature x2', fontsize=12)
axes[1].set_ylabel('Contribution to y', fontsize=12)
axes[1].set_title('Feature 2 Contribution with 95% CI', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nBayesNAM successfully learned individual feature contributions!")
print("   The model separates the additive effects: y ≈ f1(x1) + f2(x2) + bias")
print("   The 95% CI shows epistemic uncertainty about each feature's contribution.")